# DATA 202 Final Project: Does Payroll Buy Wins?
## Notebook 03: Data Quality Checks

**Students:** Asher Ring & Elliot Asquith

**What this notebook does:** Makes sure the data can be trusted before any analysis: one row per team, records that match the standings, payroll parts that add up, and an answer to the "possible problem" from our project notes (salaries below the \$780,000 league minimum).

**Reads:** `data/mlb_team_payroll_totals.csv`, `data/mlb_player_salaries.csv`, `data/mlb_standings_2026_final.csv`

**Creates:** Nothing new; it only checks and reports

**Slides it supports:** Where This Data Can Mislead; the row counts on What Each Dataset's Columns Hold and A Few Stars Carry the Bill

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("data")
IMG_DIR = Path("images")
IMG_DIR.mkdir(exist_ok=True)

teams = pd.read_csv(DATA_DIR / "mlb_team_payroll_totals.csv")
players = pd.read_csv(DATA_DIR / "mlb_player_salaries.csv")
standings_df = pd.read_csv(DATA_DIR / "mlb_standings_2026_final.csv")

print("teams:", teams.shape)
print("players:", players.shape)
print("standings:", standings_df.shape)

teams: (30, 17)
players: (2031, 12)
standings: (30, 6)


### Check 1: one row per team, and records match the official standings

In [2]:
assert len(teams) == 30 and teams["team"].is_unique

check = teams[["team", "wins", "losses"]].merge(standings_df[["team", "wins", "losses"]], on="team",
                                              how="left", validate="one_to_one", suffixes=("", "_official"))
same = (check["wins"] == check["wins_official"]) & (check["losses"] == check["losses_official"])
print("Records that match the official standings:", same.sum(), "of", len(check))
assert same.all()

Records that match the official standings: 30 of 30


### Check 2: each team's payroll categories add up to its total

In [3]:
parts = ["active_payroll", "injured_payroll", "suspended_payroll", "personal_leave_payroll",
         "minor_league_payroll", "dead_money", "retained_payroll"]
gap = teams[parts].sum(axis=1) - teams["total_payroll"]
print("Teams whose parts don't add up:", (gap != 0).sum())
assert (gap == 0).all()

Teams whose parts don't add up: 0


### Check 3: what kinds of contract lines are in the player file

In [4]:
group_counts = players["roster_group"].value_counts()
display(group_counts.rename("rows").to_frame())
print("Total contract lines:", len(players))

,rows
roster_group,
active,834
dead,502
minor,398
injured,267
retained,21
suspended,6
personal_leave,3


Total contract lines: 2031


### Check 4: salaries below the \$780,000 league minimum

The player file has two salary columns:

- `payroll_salary` = the player's **full-season** salary.
- `payroll_salary_adjusted` = the share **this team actually paid** (smaller for players who were only on the team part of the year).

So we test the minimum on `payroll_salary` for players on the major-league roster (active or injured).

In [5]:
LEAGUE_MIN = 780_000

mlb_roster = players[players["roster_group"].isin(["active", "injured"])]
below_min = mlb_roster[mlb_roster["payroll_salary"] < LEAGUE_MIN]
print("Major-league players with a full-season salary under $780,000:", len(below_min))
display(below_min[["team", "player", "roster_group", "note", "payroll_salary"]])

low_lines = players[(players["payroll_salary"] < LEAGUE_MIN) | (players["payroll_salary_adjusted"] < LEAGUE_MIN)]
print(f"\nContract lines under $780,000 in either salary column: {len(low_lines):,}")
print(low_lines["roster_group"].value_counts())

Major-league players with a full-season salary under $780,000: 2


,team,player,roster_group,note,payroll_salary
298,TOR,Jake Bloss,injured,60-DAY IL: SHOULDER,291198.0
1825,TB,Jacob Melton,injured,60-DAY IL: HAMSTRING,190443.0



Contract lines under $780,000 in either salary column: 1,126
roster_group
minor             360
dead              359
active            322
injured            80
suspended           3
personal_leave      2
Name: count, dtype: int64


*What this shows:* only two major-league players have a full-season salary under the minimum, and both are prospects on the 60-day injured list who were paid minor-league rates. The other low lines are partial-season pay, dead money, and minor-league salaries. They are real numbers, not scraping errors.

### Check 5: other things to know about (not errors)

In [6]:
blank = players["payroll_salary_adjusted"].isna().sum()
negative = (players["payroll_salary_adjusted"] < 0).sum()
repeated = players.duplicated(subset=["team", "roster_group", "player", "payroll_salary_adjusted"]).sum()
print("Lines where Spotrac shows no amount:", blank)
print("Negative amounts (money credited back to a team):", negative)
print("Lines that repeat the same player and amount:", repeated)

top = players.loc[players["payroll_salary"].idxmax()]
print(f"\nBiggest 2026 salary: {top['player']} ({top['team']}), ${top['payroll_salary'] / 1e6:.0f}M")

Lines where Spotrac shows no amount: 26
Negative amounts (money credited back to a team): 5
Lines that repeat the same player and amount: 2

Biggest 2026 salary: Shohei Ohtani (LAD), $70M


*Note:* the two repeated lines are real: Spotrac lists Dustin Harris (HOU) and Matt Krook (WSH) twice in dead money, once for each transaction. Ohtani's \$70M is listed at full value, but most of it is deferred and paid in later years. These oddities are logged in `data/data_issues_log.csv`; we leave them as Spotrac publishes them.